In [1]:
import wrds
import pandas as pd
import numpy as np
from typing import Tuple, List, Dict
from pathlib import Path

def g_cmp(d: wrds.Connection, g: str) -> pd.DataFrame:
    """Retrieves global fundamental data from Compustat Global using GVKEY.

    Output schema is normalized to use 'ni' as the net income column name
    (aliased from Compustat Global's native 'nicon') so downstream merges
    and feature engineering can be written once regardless of whether data
    came from the global or North America file.

    Note on datafmt: Compustat Global stores filings under several format
    codes. Issuers with long histories and restatements (BHP among them)
    appear under 'HIST_STD' — the historically-preserved standardized
    format — rather than the plain 'STD' used for more recent entries.
    If you run this loader against a different issuer and it returns zero
    rows, diagnose with:
        SELECT DISTINCT indfmt, datafmt, popsrc, consol, COUNT(*)
        FROM comp.g_funda WHERE gvkey = %(g)s GROUP BY 1,2,3,4

    Args:
        d: Active WRDS connection.
        g: Compustat GVKEY (zero-padded 6-character string).

    Returns:
        DataFrame sorted by datadate. Columns: datadate, curcd, at, lt,
        ni (aliased from nicon), revt. Roughly one row per fiscal year.
    """
    q = """SELECT datadate, curcd, at, lt, nicon AS ni, revt
           FROM comp.g_funda
           WHERE gvkey = %(g)s
             AND indfmt = 'INDL' AND datafmt = 'HIST_STD'
             AND popsrc = 'I' AND consol = 'C'
           ORDER BY datadate ASC"""
    return d.raw_sql(q, params={'g': g}, date_cols=['datadate'])

def g_ibs_int(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves international consensus EPS estimates from IBES summary file.

    Returns FY1 and FY2 annual EPS consensus from ibes.statsum_epsint (the
    split-adjusted international summary file). The fpi column distinguishes
    forecast horizons: '1'-'5' are annual (FY1 through FY5), 'A'-'D' are
    interim half-year estimates, '0' is long-term growth. We keep only the
    annual horizons because interims carry a different fiscal period
    convention that would break time-series merges downstream.

    Note: fpi is stored as a fixed-width character column, so values may
    carry trailing whitespace (e.g. '1 ' rather than '1'). TRIM() is
    required to match reliably — naive equality on '1' silently returns
    zero rows. Similarly, measure is trimmed for defensive consistency.

    Args:
        d: Active WRDS connection.
        t: International IBES ticker (e.g. '@BHP' for BHP Group Ltd).

    Returns:
        DataFrame sorted by statpers. Columns: statpers, fpedats, curcode,
        meanest, medest, numest, stdev, highest, lowest, fpi. Two rows per
        statpers (one for fpi='1', one for fpi='2').
    """
    q = """SELECT statpers, fpedats, curcode, meanest, medest, numest,
                  stdev, highest, lowest, TRIM(fpi) AS fpi
           FROM ibes.statsum_epsint
           WHERE ticker = %(t)s
             AND TRIM(measure) = 'EPS'
             AND TRIM(fpi) IN ('1', '2')
           ORDER BY statpers ASC"""
    return d.raw_sql(q, params={'t': t}, date_cols=['statpers', 'fpedats'])

def g_crs(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves daily stock data from CRSP Version 2 using PERMNO."""
    q = f"SELECT dlycaldt, dlyret, dlyvol FROM crsp.dsf_v2 WHERE permno = {p} AND dlycaldt >= '{s}' AND dlycaldt <= '{e}' ORDER BY dlycaldt ASC"
    return d.raw_sql(q, date_cols=['dlycaldt'])

def g_evt(d: wrds.Connection, c: str) -> pd.DataFrame:
    """Retrieves key developments from Capital IQ using CompanyID."""
    q = f"SELECT a.announcedate, a.keydeveventtypeid, b.headline FROM ciq.wrds_keydev a JOIN ciq.ciqkeydev b ON a.keydevid = b.keydevid WHERE a.companyid = {c} ORDER BY a.announcedate ASC"
    return d.raw_sql(q, date_cols=['announcedate'])

def g_ff(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves Fama-French 5-Factor daily data."""
    q = f"SELECT date, mktrf, smb, hml, rmw, cma, rf FROM ff.fivefactors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'ff_date'})
    except Exception:
        return pd.DataFrame(columns=['ff_date', 'mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf'])

def g_esg(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves ESG/Governance proxies dynamically."""
    q = f"SELECT meeting_date as as_of_date, female_directors, minority_directors FROM iss_directors_global.company_diversity WHERE ticker = '{t}' ORDER BY meeting_date ASC"
    try:
        return d.raw_sql(q, date_cols=['as_of_date'])
    except Exception:
        return pd.DataFrame(columns=['as_of_date', 'female_directors', 'minority_directors'])

def g_bdx(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves BoardEx Director Network data."""
    q = f"SELECT a.annual_report_date, AVG(b.network_size) as avg_board_network FROM boardex_row.row_wrds_org_summary a JOIN boardex_row.row_wrds_individual_networks b ON a.companyid = b.companyid WHERE a.ticker = '{t}' GROUP BY a.annual_report_date ORDER BY a.annual_report_date ASC"
    try:
        return d.raw_sql(q, date_cols=['annual_report_date'])
    except Exception:
        return pd.DataFrame(columns=['annual_report_date', 'avg_board_network'])

def g_shv(d: wrds.Connection, t: str, s: str, e: str) -> pd.DataFrame:
    """Retrieves short volume and interest data."""
    q = f"SELECT date, shortint FROM comp.sec_shortint WHERE tic = '{t}' AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'shv_date'})
    except Exception:
        return pd.DataFrame(columns=['shv_date', 'shortint'])

def g_bta(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves WRDS Beta Suite daily data."""
    q = f"SELECT date, beta FROM betasuite.beta_daily WHERE permno = {p} AND date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'bta_date'})
    except Exception:
        return pd.DataFrame(columns=['bta_date', 'beta'])

def g_fac(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves WRDS daily factors."""
    q = f"SELECT date, mkt, smb, hml, umd FROM wrdsapps.factors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'fac_date'})
    except Exception:
        return pd.DataFrame(columns=['fac_date', 'mkt', 'smb', 'hml', 'umd'])

def b_pipe(c: pd.DataFrame, i: pd.DataFrame, r: pd.DataFrame, e: pd.DataFrame, ff: pd.DataFrame, esg: pd.DataFrame, bdx: pd.DataFrame, shv: pd.DataFrame, bta: pd.DataFrame, fac: pd.DataFrame) -> pd.DataFrame:
    """Builds an aligned, forward-filled feature matrix mapping off-hours events to trading sessions.

    Args:
        c: Compustat fundamentals DataFrame.
        i: IBES estimates DataFrame.
        r: CRSP daily returns DataFrame.
        e: Capital IQ events DataFrame.
        ff: Fama-French factors DataFrame.
        esg: ISS ESG DataFrame.
        bdx: BoardEx DataFrame.
        shv: Short Volume DataFrame.
        bta: Beta Suite DataFrame.
        fac: Factors DataFrame.

    Returns:
        pd.DataFrame: Integrated and aligned feature matrix.
    """
    def _fmt(df: pd.DataFrame, col: str) -> pd.DataFrame:
        if not df.empty:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.normalize()
            return df.dropna(subset=[col]).sort_values(col)
        return df

    r = _fmt(r, 'dlycaldt')
    c = _fmt(c, 'datadate')
    i = _fmt(i, 'statpers')
    e = _fmt(e, 'announcedate')
    ff = _fmt(ff, 'ff_date')
    esg = _fmt(esg, 'as_of_date')
    bdx = _fmt(bdx, 'annual_report_date')
    shv = _fmt(shv, 'shv_date')
    bta = _fmt(bta, 'bta_date')
    fac = _fmt(fac, 'fac_date')

    if not i.empty:
        i['fpi'] = pd.to_numeric(i['fpi'], errors='coerce').astype('Int8')
        i = i.dropna(subset=['fpi'])
        i_pvt = i.groupby(['statpers', 'fpi'])[['meanest', 'medest', 'numest', 'stdev', 'highest', 'lowest', 'fpedats', 'curcode']].last().unstack()
        i_pvt.columns = [f"{col[0]}_fy{col[1]}" for col in i_pvt.columns]
        i = i_pvt.reset_index()

    m = pd.merge_asof(r, c, left_on='dlycaldt', right_on='datadate', direction='backward')

    if not i.empty: m = pd.merge_asof(m, i, left_on='dlycaldt', right_on='statpers', direction='backward')
    if not ff.empty: m = pd.merge_asof(m, ff, left_on='dlycaldt', right_on='ff_date', direction='backward')
    if not esg.empty: m = pd.merge_asof(m, esg, left_on='dlycaldt', right_on='as_of_date', direction='backward')
    if not bdx.empty: m = pd.merge_asof(m, bdx, left_on='dlycaldt', right_on='annual_report_date', direction='backward')
    if not shv.empty: m = pd.merge_asof(m, shv, left_on='dlycaldt', right_on='shv_date', direction='backward')
    if not bta.empty: m = pd.merge_asof(m, bta, left_on='dlycaldt', right_on='bta_date', direction='backward')
    if not fac.empty: m = pd.merge_asof(m, fac, left_on='dlycaldt', right_on='fac_date', direction='backward')

    if not e.empty:
        e_g = e.groupby('announcedate').agg({'keydeveventtypeid': list, 'headline': list}).reset_index().sort_values('announcedate')
        e_m = pd.merge_asof(e_g, r[['dlycaldt']], left_on='announcedate', right_on='dlycaldt', direction='forward')
        e_f = e_m.dropna(subset=['dlycaldt']).groupby('dlycaldt').agg({'keydeveventtypeid': 'sum', 'headline': 'sum'}).reset_index()
        m = pd.merge(m, e_f, on='dlycaldt', how='left')

    d_cols = ['datadate', 'statpers', 'ff_date', 'as_of_date', 'annual_report_date', 'shv_date', 'bta_date', 'fac_date']
    m.drop(columns=[col for col in d_cols if col in m.columns], inplace=True)

    return m

In [2]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [3]:
def g_sch(d: wrds.Connection, k: str) -> pd.DataFrame:
    """Lists schema.table pairs whose schema name matches a keyword."""
    q = "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema LIKE %(k)s ORDER BY table_schema, table_name"
    return d.raw_sql(q, params={'k': f'%{k}%'})

def g_col(d: wrds.Connection, schema: str, table: str) -> pd.DataFrame:
    """Lists columns and their data types for a given schema.table."""
    q = "SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = %(s)s AND table_name = %(t)s ORDER BY ordinal_position"
    return d.raw_sql(q, params={'s': schema, 't': table})

In [2]:
usr      = "zackienzle1"
gvkey    = "100712"
permno   = 23169
ibes_tic = "@WPL"
ciq_id   = "873963"
us_tic   = "WDS"
st       = "2000-01-01"
ed       = "2026-04-10"

db = wrds.Connection(wrds_username=usr)

Loading library list...
Done


In [5]:
df_cmp = g_cmp(db, gvkey)
df_ibs = g_ibs_int(db, ibes_tic)
df_crs = g_crs(db, permno, st, ed)
df_evt = g_evt(db, ciq_id)
df_ff = g_ff(db, st, ed)
df_esg = g_esg(db, ibes_tic)
df_bdx = g_bdx(db, ibes_tic)
df_shv = g_shv(db, us_tic, st, ed)
df_bta = g_bta(db, permno, st, ed)
df_fac = g_fac(db, st, ed)

db.close()

df_main = b_pipe(df_cmp, df_ibs, df_crs, df_evt, df_ff, df_esg, df_bdx, df_shv, df_bta, df_fac)

In [6]:
df_main

,dlycaldt,dlyret,dlyvol,curcd,at,lt,ni,revt,meanest_fy1,meanest_fy2,...,curcode_fy1,curcode_fy2,mktrf,smb,hml,rmw,cma,rf,keydeveventtypeid,headline
0,2022-06-02,<NA>,1422570.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,USD,USD,0.0208,0.004,-0.0208,-0.0042,-0.0153,0.0,"[95, 95, 95, 95, 95, 80, 95, 95, 95, 95, 80, 8...",[Woodside Petroleum Ltd.(ASX:WPL) added to FTS...
1,2022-06-03,-0.006479,1233276.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,USD,USD,-0.0163,0.0103,0.0055,-0.0081,0.0046,0.0,NaN,NaN
2,2022-06-06,0.033044,1651320.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,USD,USD,0.0033,0.0004,0.0066,0.0047,0.0017,0.0,NaN,NaN
3,2022-06-07,0.014731,2265246.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,USD,USD,0.01,0.002,-0.0015,-0.0157,0.0065,0.0,NaN,NaN
4,2022-06-08,0.037744,1899037.0,USD,26474.0,12245.0,1983.0,6962.0,3.24,2.54,...,USD,USD,-0.0102,-0.0027,-0.0052,-0.007,-0.0025,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
894,2025-12-24,-0.003876,424350.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,USD,USD,0.0029,0.0007,0.0001,-0.0005,0.0023,0.0002,NaN,NaN
895,2025-12-26,-0.003891,776072.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,USD,USD,-0.0006,-0.0022,0.0009,0.0057,0.0024,0.0002,NaN,NaN
896,2025-12-29,0.010417,694558.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,USD,USD,-0.0041,-0.0017,0.0007,0.0032,0.0002,0.0002,[23],[Woodside Energy Announces Production Mileston...
897,2025-12-30,0.006443,543998.0,USD,61264.0,25111.0,3573.0,13179.0,1.19,0.79,...,USD,USD,-0.002,-0.0049,0.0028,0.0036,0.0013,0.0002,NaN,NaN


In [7]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   dlycaldt           899 non-null    datetime64[ns]
 1   dlyret             898 non-null    Float64       
 2   dlyvol             899 non-null    Float64       
 3   curcd              899 non-null    string        
 4   at                 899 non-null    Float64       
 5   lt                 899 non-null    Float64       
 6   ni                 899 non-null    Float64       
 7   revt               899 non-null    Float64       
 8   meanest_fy1        899 non-null    Float64       
 9   meanest_fy2        899 non-null    Float64       
 10  medest_fy1         899 non-null    Float64       
 11  medest_fy2         899 non-null    Float64       
 12  numest_fy1         899 non-null    Float64       
 13  numest_fy2         899 non-null    Float64       
 14  stdev_fy1 

In [8]:
def s_csv(d_m: Dict[str, pd.DataFrame], d_p: str = "../data") -> None:
    """Saves multiple DataFrames to CSV efficiently."""
    p = Path(d_p)
    p.mkdir(parents=True, exist_ok=True)
    for k, v in d_m.items():
        if not v.empty:
            v.to_csv(p / f"{k}.csv", index=False, chunksize=100000)

In [10]:
d_out = {
    "compustat_fundamentals": df_cmp,
    "ibes_eps_summary": df_ibs,
    "crsp_daily_prices": df_crs,
    "capitaliq_key_developments": df_evt,
    "fama_french_5f_daily": df_ff,
    "short_interest": df_shv,
    "integrated_feature_matrix": df_main
}

s_csv(d_out)
print(f"Pipeline executed. Main matrix shape: {df_main.shape}")

Pipeline executed. Main matrix shape: (899, 32)


In [ ]:
# import wrds
# import pandas as pd

# def ext_evt(db: wrds.Connection, out_p: str = "ciq_all_event_types.csv") -> pd.DataFrame:
#     q = """
#         SELECT keydeveventtypeid, eventtypename AS event_name
#         FROM ciq.ciqeventtype
#         ORDER BY keydeveventtypeid
#     """
#     df = db.raw_sql(q)
#     df.to_csv(out_p, index=False)
#     return df

# def d_cov(db: wrds.Connection, s: str, t: str, d_c: str, i_c: str, i_v: str) -> pd.DataFrame:
#     q = f"""
#         SELECT MIN({d_c}) AS min_date, MAX({d_c}) AS max_date, COUNT(1) AS n_obs
#         FROM {s}.{t}
#         WHERE {i_c} = '{i_v}'
#     """
#     return db.raw_sql(q)

# def d_lnk(db: wrds.Connection, gvkey: str) -> pd.DataFrame:
#     q = f"""
#         SELECT lpermno, linkdt, linkenddt, linktype, linkprim
#         FROM crsp.ccmxpf_lnkhist
#         WHERE gvkey = '{gvkey}'
#         ORDER BY linkdt ASC
#     """
#     return db.raw_sql(q)

In [ ]:
# df_evt = ext_evt(db)
# print(df_evt.head().to_string(index=False))
# print("\n")

# df_cov = d_cov(db, "ciq", "wrds_keydev", "announcedate", "companyid", ciq_id)
# print(df_cov.to_string(index=False))
# print("\n")

# df_lnk = d_lnk(db, gvkey)
# print(df_lnk.to_string(index=False))

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "ciq.ciqeventtype" does not exist
LINE 3:         FROM ciq.ciqeventtype
                     ^

[SQL: 
        SELECT keydeveventtypeid, eventtypename AS event_name
        FROM ciq.ciqeventtype
        ORDER BY keydeveventtypeid
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)